# Filtering 60 k Dataset based on train distribution

My objective here is to study the distribution of topics in train.csv and filter the 60 k Dataset to match the same distribution. For acomplish this task I used Generative Pseudo Labeling (GPL) to train a embedding model in wikipedia STEM related pages, then I used the trained model and BERTopic library to cluster questions of train.csv in topics, then apply the same classification on 60 k dataset and finally resampled the 60 K dataset to match the topic distribution of the train.csv


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/60k-data-with-context-v2/all_12_with_context2.csv
/kaggle/input/60k-data-with-context-v2/train_with_context2.csv
/kaggle/input/60k-data-with-context-v2/sources.txt
/kaggle/input/kaggle-llm-science-exam/sample_submission.csv
/kaggle/input/kaggle-llm-science-exam/train.csv
/kaggle/input/kaggle-llm-science-exam/test.csv


In [2]:
#https://arxiv.org/pdf/2303.09128.pdf

In [3]:
!pip install bertopic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.4/143.4 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 57.0 MB/s eta 0:00:00:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for hdbscan: filename=hdbscan-0.8.33-cp310-cp310-linux_x86_64.whl size=819467 sha256=17dd0d66836226bca72b0896ea646ffe3663507aacbaddc9afde6e22ca50ffc6
  Stored in directory: /root/.cache/pip/wheels/75/0b/3b/dc4f60b7cc455efaefb62883a7483e76f09d06ca81cf87d610
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125926 sha256=9b18bfa1f614143998d7693c3ec03beb5ee8afa49543c53da319d03c79ddc13c
  Stored in directory: /root/.cache/pip/wheels/62/f2/10/1e606fd5f02395388f74e7462910fe851042f97238cbbd902f
Successfully

In [4]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.representation import MaximalMarginalRelevance

from hdbscan import HDBSCAN
from sklearn.cluster import KMeans
from umap import UMAP

from sentence_transformers import SentenceTransformer

/opt/conda/lib/python3.10/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/opt/conda/lib/python3.10/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/opt/conda/lib/python3.10/site-packages/umap/distances.py:1086: NumbaDeprecationWarning: The 'nopython' keyword argume

In [5]:
train = pd.read_csv('/kaggle/input/60k-data-with-context-v2/all_12_with_context2.csv')
test = pd.read_csv('/kaggle/input/kaggle-llm-science-exam/test.csv')

In [6]:
train['sentence'] = train['prompt'].astype(str)+' '+train['A'].astype(str)+' '+train['B'].astype(str)+' '+train['C'].astype(str)+' '+train['D'].astype(str)+' '+train['E'].astype(str)
test['sentence'] = test['prompt']+' '+test['A']+' '+test['B']+' '+test['C']+' '+test['D']+' '+test['E']

In [7]:
train.head()

,prompt,context,A,B,C,D,E,answer,source,sentence
0,"In relation to Eunice Fay McKenzie's career, w...","Eunice Fay McKenzie (February 19, 1918 – April...",McKenzie showcased her singing talents in nume...,McKenzie is primarily remembered for her starr...,McKenzie gained recognition for her role as a ...,McKenzie's collaborations with director Blake ...,McKenzie's successful career in sound films co...,B,1,"In relation to Eunice Fay McKenzie's career, w..."
1,How does Modified Newtonian Dynamics (MOND) im...,The presence of a clustered thick disk-like co...,MOND is a theory that increases the discrepanc...,MOND explains the missing baryonic mass in gal...,MOND is a theory that reduces the observed mis...,MOND is a theory that eliminates the observed ...,MOND's impact on the observed missing baryonic...,E,1,How does Modified Newtonian Dynamics (MOND) im...
2,Which of the following statements accurately d...,Woody Hartman is a retired American soccer goa...,Ray Montgomerie is a former footballer who pla...,Ray Montgomerie is a former footballer who pla...,Ray Montgomerie is a former footballer who pla...,Ray Montgomerie is a former footballer who pla...,Ray Montgomerie is a former footballer who pla...,B,1,Which of the following statements accurately d...
3,What is the significance of the Museum of the ...,The Museum of the Occupation of Latvia () is a...,The Museum of the Occupation of Latvia is a me...,The Museum of the Occupation of Latvia showcas...,The Museum of the Occupation of Latvia was est...,The Museum of the Occupation of Latvia primari...,The Museum of the Occupation of Latvia is a mu...,C,1,What is the significance of the Museum of the ...
4,What was the previous name of the Christian Sc...,It was named the Evangelical School for the De...,The Christian School for the Deaf (CSD),The Christian School for the Blind (CSB),The Evangelical School and Chapel for the Deaf...,The Evangelical School for the Deaf (ESD),The Evangelical School for the Blind (ESB),D,1,What was the previous name of the Christian Sc...


# Generate topics based on train CSV

In [8]:
embedding_model = SentenceTransformer('/kaggle/input/llmse-generative-pseudo-labeling/output/')

ValueError: Path /kaggle/input/llmse-generative-pseudo-labeling/output/ not found

In [ ]:
representation_model = KeyBERTInspired()
#representation_model = MaximalMarginalRelevance(diversity=0.3)

#cluster_model = KMeans(n_clusters=20)
hdbscan_model = HDBSCAN(min_cluster_size=3, min_samples=3, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

umap_model = UMAP(n_neighbors=3, n_components=10, min_dist=0.0, metric='cosine')

topic_model = BERTopic(
    representation_model=representation_model,
    embedding_model=embedding_model,
    hdbscan_model=hdbscan_model,
    umap_model=umap_model
)

In [ ]:
topics, probs = topic_model.fit_transform(test['sentence'])
test['topics'] = topics

In [ ]:
topic_model.get_topic_info()

In [ ]:
for idx, row in topic_model.get_topic_info().iterrows():
    print('Topic id ', row['Topic'])
    print('Representation: ', row['Representation'])

In [ ]:
topic_model.visualize_documents(test['prompt'])

In [ ]:
topic_model.visualize_topics()

In [ ]:
topics, probs = topic_model.transform(train['sentence'])
train['topics'] = topics

# Train CSV Topics Distribution

In [ ]:
import seaborn as sns

sns.displot(test, x='topics', kind='kde')

# 60 k Dataset Topics Distribution

In [ ]:
sns.displot(train, x='topics', kind='kde')

# Resampled 60 k Dataset

In [ ]:
topic_probabilities = test.groupby(['topics']).size().reset_index(name='count')
topic_probabilities['prob'] = topic_probabilities['count']/topic_probabilities['count'].sum()
topic_probabilities.head()

In [ ]:
train = pd.merge(
    train,
    topic_probabilities[['topics','prob']],
    how='left',
    on='topics'
)

train.head()

In [ ]:
topic_counts = train.groupby('topics').size().reset_index(name='group_count')
train = pd.merge(
    train,
    topic_counts,
    how='left',
    on='topics'
)
train.head()

In [ ]:
train['item_prob'] = train['prob']/train['group_count']
train.head()

In [ ]:
train['item_prob'].sum()

In [ ]:
new_train_df_index = np.random.choice(
    train.index,
    size=5_000,
    replace=False,
    p=train['item_prob'].to_list()
)

In [ ]:
new_train_df = train.iloc[new_train_df_index]

In [ ]:
sns.displot(new_train_df, x='topics', kind='kde')

In [ ]:
new_train_df[['prompt','context','A','B','C','D','E','answer']].to_csv('train.csv', index=False)